# D7. 학습량 200만 — 고친 정답(split_v3b)으로 (Colab A100 + 높은 RAM)

**왜**: 120만이 60만보다 dev +2.16점(정답 수정 몫 약 0.34 제외해도 +1.8점) 올라 미리 정한 기준(0.7점)을 넘었다 → 200만에서도 오르는지 본다.
**미리 정한 판단 기준**: 120만(dev 0.7875) 대비 dev F1이 **0.7점 이상** 오르면 더 늘리는 것을 검토, 아니면 멈춘다.

**데이터**: data/split_v3b/train_2000k.jsonl = 120만 세트(앞부분 그대로) + 새 80만(카테고리 17개 균등, 고친 정답 규칙, 평가·골드셋·중복 제외). 독립 검증기 통과.
**조건**: klue/roberta-base, 3에폭, 학습률 5e-5, 배치 16, 최대 길이 512, 시드 42 (120만과 같음).
**시간(추정)**: 120만 실측 3.45시간 × 200/120 ≈ **약 5.8시간** + 데이터 변환 약 7분 + 검증 약 10분.
**끊겨도 이어서**: 5,000스텝마다 내 드라이브 ner_ckpt에 저장. 끊기면 "모두 실행"을 다시 누르면 마지막 저장부터 이어서 학습.
**런타임**: A100, **높은 RAM**(200만 문장을 읽을 때 약 10GB). 끝나면 결과 확인 후 자동 반납.


## 0. 공통 준비 (모든 노트북 같음)
Colab → 런타임 → 런타임 유형 변경 → GPU(T4 이상). 구글 드라이브를 연결하면 이 PC의 프로젝트 폴더가 `/content/drive/Othercomputers/내 PC/...` 아래에 보인다. 경로가 다르면 왼쪽 파일 탭에서 찾아 `PROJECT` 만 고친다.

공용 코드 `src/ner_lib.py`(학습)와 `src/evaluate.py`(팀 공용 채점기)를 불러온다. 정렬·복원·채점 규칙이 모든 노트북에서 같아야 비교가 공정하기 때문이다.

**저장되는 것**: 실험마다 `reports/results/{실험}.json`(에폭별 dev 점수 포함), `preds/`(예측), `scores/`(채점), `errors/`(오답), 그리고 `runs.jsonl`(실행 기록 한 줄). 그림은 R0 노트북이 이 파일들로 그린다.
**끊겼을 때**: 같은 셀을 다시 실행하면 끝난 실험은 건너뛰고, 진행 중이던 실험은 마지막 에폭 체크포인트부터 이어서 학습한다.

**최종 가중치(10/23부터 항상 저장)**: 학습이 끝나면 예측·채점보다 먼저 `내 드라이브/ner_models/{실험}/` 에 가중치(model.safetensors 또는 crf_model_state.pt)와 토크나이저를 저장하고, 다시 읽어 학습된 값과 같은지 확인한다. 확인에 실패하면 체크포인트를 지우지 않고 멈춘다. 런타임 반납 전에도 한 번 더 확인한다.

In [ ]:
!pip -q install -U transformers datasets accelerate pytorch-crf kiwipiepy
import sys, os, json, torch
from google.colab import drive
drive.mount("/content/drive")
PROJECT = "/content/drive/Othercomputers/내 PC/NLP/프로젝트1/공통2_개체명인식_NER"
sys.path.insert(0, f"{PROJECT}/src")
import importlib, ner_lib; importlib.reload(ner_lib)
MODELS = "/content/drive/MyDrive/ner_models"   # 최종 가중치(실험마다 폴더). 학습 직후 저장하고 다시 읽어 확인한다. 지우지 말 것
ner_lib.DEFAULTS.update(model_dir=MODELS)
from ner_lib import run_experiment, load_results, results_table, upper_bound, load_jsonl, prepare_aihub_model
SPLIT   = f"{PROJECT}/data/split_v1"          # dev 1만 / test 1만 (모든 버전 공통)
TRAIN   = f"{PROJECT}/data/split_v3/train.jsonl"   # 팀 기준 학습 데이터: 30만 (2026-10-22 결정)
SUFFIX  = "_300k"                              # 30만으로 학습한 실험 이름 꼬리표
RESULTS = f"{PROJECT}/reports/results"        # 결과 json이 모이는 곳 (팀원 모두 같은 폴더)
CKPT    = "/content/drive/MyDrive/ner_ckpt"    # 에폭 체크포인트. 끊겨도 이어서 학습. '내 드라이브'라 PC로 동기화되지 않음
MEMBER  = "sejin_choi"                          # ← 실행한 사람 이름, 영어로 (결과 기록 runs.jsonl에 남음)
ner_lib.DEFAULTS.update(ckpt_dir=CKPT, member=MEMBER)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음")

In [ ]:
# 한글 글꼴 (그래프 글자가 네모로 깨지지 않게)
!apt-get -qq install -y fonts-nanum > /dev/null
import matplotlib.pyplot as plt, matplotlib.font_manager as fm, glob as _g
for f in _g.glob("/usr/share/fonts/truetype/nanum/*.ttf"): fm.fontManager.addfont(f)
plt.rc("font", family="NanumGothic"); plt.rc("axes", unicode_minus=False)
FIG = f"{PROJECT}/reports/figures"; os.makedirs(FIG, exist_ok=True)
def savefig(name):
    plt.tight_layout(); plt.savefig(f"{FIG}/{name}.png", dpi=150); plt.show(); print("저장:", f"{FIG}/{name}.png")

In [ ]:
ner_lib.DEFAULTS["stage"] = "D7_학습량200만"
V = f"{PROJECT}/data/split_v3b"
COMMON = dict(dev=f"{V}/dev.jsonl", test=f"{V}/test.jsonl", out_dir=RESULTS, model="klue/roberta-base",
              max_len=512, lr=5e-5, epochs=3, batch=16, seed=42, save_steps=5000)
RUNS = {"D7_v3b_2000k": dict(train=f"{V}/train_2000k.jsonl")}
print("GPU:", torch.cuda.get_device_name(0), "| bf16:", torch.cuda.is_bf16_supported())
import psutil; print(f"RAM {psutil.virtual_memory().total/1e9:.0f}GB")


## 1. 시작 전 확인 — 파일과 데이터 검증
파일이 드라이브에 다 올라왔는지 보고, **독립 검증기(src/validate_train.py)를 다시 돌린다. 하나라도 FAIL이면 여기서 멈춘다.**


In [ ]:
need = [COMMON["dev"], COMMON["test"], RUNS["D7_v3b_2000k"]["train"], f"{V}/train_1200k.jsonl", f"{PROJECT}/data/goldset/goldset_master.jsonl"]
for f in need:
    ok = os.path.exists(f); print("있음" if ok else "없음!!", f.replace(PROJECT, "..."), f"{os.path.getsize(f)/1e6:.1f}MB" if ok else "")
assert all(os.path.exists(f) for f in need), "파일이 아직 드라이브에 없습니다(PC 동기화 대기)"
import subprocess
r = subprocess.run(["python", "src/validate_train.py", "--file", "data/split_v3b/train_2000k.jsonl", "--base", "data/split_v3b/train_1200k.jsonl", "--n", "2000000"],
                   cwd=PROJECT, capture_output=True, text=True)
print(r.stdout[-3000:]); print(r.stderr[-1000:])
assert r.returncode == 0, "데이터 검증 FAIL — 학습하지 않습니다"


## 2. 실행
끊기면 0번 셀부터 다시 실행한다. 진행 중이던 실험은 '내 드라이브/ner_ckpt'의 마지막 저장(5,000스텝마다)부터 이어서 학습한다.


In [ ]:
for exp, v in RUNS.items():
    run_experiment({**COMMON, **v, "exp": exp})


## 3. 판단 — 120만 대비
120만(dev 0.7875 / test 0.7773, 같은 고친 정답·같은 v3b dev·test)과 비교한다.


In [ ]:
r = json.load(open(f"{RESULTS}/D7_v3b_2000k.json", encoding="utf-8"))
d, t = r["dev"]["ALL"]["f1"], r["test"]["ALL"]["f1"]
print(f"200만: dev {d} / test {t} / test macro {r['test']['MACRO_F1']} | 학습 {r['train_sec']/3600:.2f}시간")
print("에폭별 dev:", [(h["epoch"], h["dev_f1"]) for h in r["history"]])
diff = (d - 0.7875) * 100
print(f"120만(dev 0.7875) 대비 dev {diff:+.2f}점 →", "더 늘리는 것 검토" if diff >= 0.7 else "여기서 멈춤")


## 4. 저장 확인 후 런타임 반납
결과 파일이 모두 드라이브에 있으면 저장을 마무리하고 런타임을 반납한다. 하나라도 없으면 멈춘다.


In [ ]:
ner_lib.verify_saved_models()   # 이번에 다룬 실험의 최종 가중치가 하나라도 없으면 여기서 멈추고 반납하지 않음
print("가중치 위치:", MODELS)
missing = []
for exp in RUNS:
    for f in (f"{RESULTS}/{exp}.json", f"{RESULTS}/scores/{exp}_dev.json", f"{RESULTS}/scores/{exp}_test.json",
              f"{RESULTS}/preds/{exp}_dev.jsonl", f"{RESULTS}/preds/{exp}_test.jsonl",
              f"{RESULTS}/errors/{exp}_dev.jsonl", f"{RESULTS}/errors/{exp}_test.jsonl"):
        if not os.path.exists(f): missing.append(f.replace(RESULTS, "results"))
if missing:
    print("저장 안 된 파일이 있어 반납하지 않습니다:", missing[:10])
else:
    print("결과 파일 모두 저장됨. 드라이브 저장을 마무리하고 런타임을 반납합니다.")
    drive.flush_and_unmount()
    from google.colab import runtime
    runtime.unassign()